**Imports**


In [19]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import copy
import random
import math
import warnings
import json

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

import plotly.graph_objects as go
import plotly.subplots as sp

from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")

In [2]:
from google.colab import files

uploaded = files.upload()


Saving Alcohol sales.csv to Alcohol sales.csv


**Configuration**

In [3]:
CSV_PATH = "Alcohol sales.csv"

DATE_COL = "observation_date"
TARGET_COL = "S4248SM144NCEN"

LOOKBACK = 36

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

K_VALUES = [0, 1, 2, 3, 4, 5]

SEEDS = [42, 123, 2026]

EPOCHS = 150
PATIENCE = 25

BATCH_SIZE = 32

HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.15

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

RUN_ROLLING_VALIDATION = True
ROLLING_FOLDS = 3
ROLLING_HORIZON = 12

NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("ADVANCED LSTM + FOURIER FORECASTING")
print("=" * 65)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


ADVANCED LSTM + FOURIER FORECASTING
Device: cpu
PyTorch: 2.11.0+cpu


**REPRODUCIBILITY**

In [5]:
def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

**LOAD** **AND** **PREPARE** **DATA**

In [6]:
df = pd.read_csv(CSV_PATH)

df[DATE_COL] = pd.to_datetime(df[DATE_COL])

df[TARGET_COL] = pd.to_numeric(
    df[TARGET_COL],
    errors="coerce"
)

df = (
    df[[DATE_COL, TARGET_COL]]
    .dropna()
    .sort_values(DATE_COL)
    .drop_duplicates(subset=DATE_COL)
    .reset_index(drop=True)
)

if (df[TARGET_COL] <= 0).any():
    raise ValueError(
        "Sales must be positive because logarithmic transformation is used."
    )

dates = pd.DatetimeIndex(df[DATE_COL])

sales = df[TARGET_COL].to_numpy(dtype=np.float64)

n = len(sales)

if n <= LOOKBACK + 36:
    raise ValueError(
        "Dataset is too small for the selected lookback and validation setup."
    )

train_end = int(n * TRAIN_RATIO)

val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

test_start = val_end

dev_end = val_end

print("\nDataset shape:", df.shape)

print(
    f"Train: {dates[0].date()} -> "
    f"{dates[train_end-1].date()} "
    f"({train_end} observations)"
)

print(
    f"Validation: {dates[train_end].date()} -> "
    f"{dates[val_end-1].date()} "
    f"({val_end-train_end} observations)"
)

print(
    f"Test: {dates[test_start].date()} -> "
    f"{dates[-1].date()} "
    f"({n-test_start} observations)"
)


Dataset shape: (398, 2)
Train: 1992-01-01 -> 2015-02-01 (278 observations)
Validation: 2015-03-01 -> 2020-02-01 (60 observations)
Test: 2020-03-01 -> 2025-02-01 (60 observations)


**DATA OVERVIEW**

In [7]:
fig = sp.make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Monthly Alcohol Sales",
        "Log-transformed Sales"
    )
)

fig.add_trace(
    go.Scatter(
        x=dates,
        y=sales,
        name="Sales",
        mode="lines"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=dates,
        y=np.log(sales),
        name="Log Sales",
        mode="lines"
    ),
    row=2,
    col=1
)

fig.add_vline(
    x=dates[train_end],
    line_dash="dash",
    line_color="green"
)

fig.add_vline(
    x=dates[val_end],
    line_dash="dash",
    line_color="red"
)

fig.update_layout(
    title="Alcohol Sales Dataset Overview",
    template="plotly_white",
    height=650,
    hovermode="x unified"
)

fig.show()

**TREND & RESIDUAL DECOMPOSITION**

In [8]:
sales_log = np.log(sales)

time_index = np.arange(n, dtype=np.float64)


def fit_preprocessor(log_values, fit_end):

    t = np.arange(len(log_values), dtype=np.float64)

    coef = np.polyfit(
        t[:fit_end],
        log_values[:fit_end],
        deg=1
    )

    trend = np.polyval(coef, t)

    residual = log_values - trend

    scaler = StandardScaler()

    scaler.fit(
        residual[:fit_end].reshape(-1, 1)
    )

    residual_scaled = scaler.transform(
        residual.reshape(-1, 1)
    ).flatten()

    trend_min = trend[:fit_end].min()

    trend_max = trend[:fit_end].max()

    trend_normalized = (
        (trend - trend_min) /
        (trend_max - trend_min + 1e-8)
    )

    return {
        "coef": coef,
        "trend": trend,
        "residual": residual,
        "scaler": scaler,
        "residual_scaled": residual_scaled,
        "trend_min": trend_min,
        "trend_max": trend_max,
        "trend_normalized": trend_normalized
    }


initial_prep = fit_preprocessor(
    sales_log,
    train_end
)

trend_log = initial_prep["trend"]

residual = initial_prep["residual"]


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=dates,
        y=sales_log,
        name="Log Sales"
    )
)

fig.add_trace(
    go.Scatter(
        x=dates,
        y=trend_log,
        name="Linear Trend",
        line=dict(dash="dash")
    )
)

fig.update_layout(

    title="Log Sales and Estimated Trend",
    template="plotly_white",
    height=450
)

fig.show()


**FOURIER FEATURE GENERATION**

In [9]:
def create_fourier_features(
    dates,
    K=3,
    period=12
):

    t = np.arange(len(dates), dtype=np.float64)

    features = []

    for k in range(1, K + 1):

        features.append(
            np.sin(
                2 * np.pi * k * t / period
            )
        )

        features.append(
            np.cos(
                2 * np.pi * k * t / period
            )
        )

    if len(features) == 0:

        return np.empty(
            (len(dates), 0),
            dtype=np.float32
        )

    return np.column_stack(features).astype(
        np.float32
    )


**FEATURE MATRIX CONSTRUCTION**

In [10]:
def build_features(prep, dates, K):

    fourier = create_fourier_features(
        dates,
        K=K,
        period=12
    )

    features = np.column_stack([
        prep["residual_scaled"],
        fourier,
        prep["trend_normalized"]
    ])

    return features.astype(np.float32)


**SEQUENCE GENERATION**

In [11]:

def create_sequences(
    features,
    targets,
    lookback
):

    X = []
    y = []
    indices = []

    for i in range(lookback, len(features)):

        X.append(
            features[i-lookback:i]
        )

        y.append(
            targets[i]
        )

        indices.append(i)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32).reshape(-1, 1),
        np.asarray(indices)
    )

**MODEL DEFINITION**

In [12]:

class LSTMRegressor(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=2,
        dropout=0.15
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Sequential(

            nn.Linear(
                hidden_size,
                32
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        last_hidden = output[:, -1, :]

        return self.fc(last_hidden)

**METRICS**

In [13]:

def calculate_metrics(actual, predicted):

    actual = np.asarray(actual).flatten()

    predicted = np.asarray(predicted).flatten()

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    mae = mean_absolute_error(
        actual,
        predicted
    )

    mape = np.mean(
        np.abs(
            (actual - predicted) /
            np.maximum(np.abs(actual), 1e-8)
        )
    ) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape
    }


**SALES RECONSTRUCTION**

In [14]:

def reconstruct_sales(
    residual_scaled,
    prep,
    indices
):

    residual_log = prep["scaler"].inverse_transform(
        np.asarray(residual_scaled).reshape(-1, 1)
    ).flatten()

    trend_values = prep["trend"][indices]

    return np.exp(
        residual_log + trend_values
    )

**MODEL TRAINING FUNCTION**

In [15]:
def train_lstm(
    K,
    seed,
    fit_end,
    validation_end,
    epochs=EPOCHS,
    fixed_epochs=None,
    verbose=False
):

    set_seed(seed)

    prep = fit_preprocessor(
        sales_log,
        fit_end
    )

    features = build_features(
        prep,
        dates,
        K
    )

    X_all, y_all, indices = create_sequences(
        features,
        prep["residual_scaled"],
        LOOKBACK
    )

    train_mask = indices < fit_end

    val_mask = (
        (indices >= fit_end) &
        (indices < validation_end)
    )

    X_train = X_all[train_mask]

    y_train = y_all[train_mask]

    X_val = X_all[val_mask]

    y_val = y_all[val_mask]

    val_indices = indices[val_mask]

    if len(X_train) == 0:
        raise ValueError("No training sequences available.")

    train_ds = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS
    )

    model = LSTMRegressor(
        input_size=features.shape[1],
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(DEVICE)

    criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=6,
        min_lr=1e-5
    )

    history = {
        "train": [],
        "val": [],
        "lr": []
    }

    best_loss = float("inf")

    best_epoch = 0

    best_state = None

    patience_counter = 0

    if fixed_epochs is not None:

        total_epochs = fixed_epochs

    else:

        total_epochs = epochs

    if len(X_val) > 0:

        X_val_t = torch.from_numpy(
            X_val
        ).to(DEVICE)

        y_val_t = torch.from_numpy(
            y_val
        ).to(DEVICE)

    else:

        X_val_t = None

        y_val_t = None

    for epoch in range(1, total_epochs + 1):

        model.train()

        running_loss = 0.0

        for bx, by in train_loader:

            bx = bx.to(DEVICE)

            by = by.to(DEVICE)

            optimizer.zero_grad()

            predictions = model(bx)

            loss = criterion(
                predictions,
                by
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            running_loss += (
                loss.item() * bx.size(0)
            )

        train_loss = (
            running_loss / len(train_ds)
        )

        history["train"].append(train_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        if len(X_val) > 0:

            model.eval()

            with torch.no_grad():

                val_predictions = model(
                    X_val_t
                )

                val_loss = criterion(
                    val_predictions,
                    y_val_t
                ).item()

            scheduler.step(val_loss)

            history["val"].append(val_loss)

            improved = (
                val_loss < best_loss - 1e-7
            )

            if improved:

                best_loss = val_loss

                best_epoch = epoch

                best_state = copy.deepcopy(
                    model.state_dict()
                )

                patience_counter = 0

            else:

                patience_counter += 1

            if verbose and (
                epoch % 10 == 0 or improved
            ):

                print(
                    f"Epoch {epoch:03d} | "
                    f"Train {train_loss:.5f} | "
                    f"Val {val_loss:.5f} | "
                    f"LR {current_lr:.2e}"
                )

            if patience_counter >= PATIENCE:

                if verbose:

                    print(
                        f"Early stopping at epoch {epoch}"
                    )

                break

        else:

            history["val"].append(np.nan)

            best_epoch = epoch

        history["lr"].append(
            current_lr
        )

    if best_state is not None:

        model.load_state_dict(best_state)

    model.eval()

    return {
        "model": model,
        "prep": prep,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_loss": best_loss,
        "features": features
    }


**PREDICTION FUNCTION**

In [16]:
def predict_model(
    model,
    features,
    prep,
    start_index,
    end_index
):

    model.eval()

    X = []

    target_indices = []

    for i in range(
        max(LOOKBACK, start_index),
        end_index
    ):

        X.append(
            features[i-LOOKBACK:i]
        )

        target_indices.append(i)

    X = np.asarray(
        X,
        dtype=np.float32
    )

    X_t = torch.from_numpy(X).to(DEVICE)

    with torch.no_grad():

        predictions = model(
            X_t
        ).cpu().numpy().flatten()

    target_indices = np.asarray(
        target_indices
    )

    reconstructed = reconstruct_sales(
        predictions,
        prep,
        target_indices
    )

    return (
        predictions,
        reconstructed,
        target_indices
    )


**SEASONAL NAIVE BASELINE**

In [17]:
def seasonal_naive(
    values,
    indices,
    period=12
):

    return values[
        indices - period
    ]

**FOURIER LINEAR REGRESSION BASELINE**

In [18]:

def fourier_regression(
    K,
    fit_end,
    prediction_indices
):

    t = np.arange(n, dtype=np.float64)

    fourier = create_fourier_features(
        dates,
        K=K,
        period=12
    )

    X = np.column_stack([
        t,
        fourier
    ])

    model = LinearRegression()

    model.fit(
        X[:fit_end],
        sales_log[:fit_end]
    )

    predictions_log = model.predict(
        X[prediction_indices]
    )

    predictions = np.exp(
        predictions_log
    )

    return predictions, model


**EXPERIMENT 1: BASELINES**

In [20]:

print("\n" + "=" * 65)
print("BASELINE EXPERIMENTS")
print("=" * 65)

baseline_results = []

val_indices = np.arange(
    train_end,
    val_end
)

test_indices = np.arange(
    test_start,
    n
)

for split_name, indices in [
    ("Validation", val_indices),
    ("Test", test_indices)
]:

    pred = seasonal_naive(
        sales,
        indices,
        period=12
    )

    actual = sales[indices]

    metrics = calculate_metrics(
        actual,
        pred
    )

    baseline_results.append({
        "Model": "Seasonal Naive",
        "K": None,
        "Split": split_name,
        **metrics
    })

for K in K_VALUES:

    for split_name, indices in [
        ("Validation", val_indices),
        ("Test", test_indices)
    ]:

        pred, _ = fourier_regression(
            K,
            train_end,
            indices
        )

        metrics = calculate_metrics(
            sales[indices],
            pred
        )

        baseline_results.append({
            "Model": "Fourier Regression",
            "K": K,
            "Split": split_name,
            **metrics
        })


baseline_df = pd.DataFrame(
    baseline_results
)

display(baseline_df)




BASELINE EXPERIMENTS


,Model,K,Split,RMSE,MAE,MAPE
0,Seasonal Naive,NaN,Validation,738.962369,601.816667,4.874212
1,Seasonal Naive,NaN,Test,1120.557406,861.033333,5.613525
2,Fourier Regression,0.0,Validation,1756.547881,1312.110844,11.786759
3,Fourier Regression,0.0,Test,2393.170544,1755.398114,12.497047
4,Fourier Regression,1.0,Validation,1683.741347,1386.555368,11.972244
5,Fourier Regression,1.0,Test,2257.506466,1808.616099,12.436936
6,Fourier Regression,2.0,Validation,1498.904270,1297.419096,11.025068
7,Fourier Regression,2.0,Test,2139.640357,1730.996295,11.718374
8,Fourier Regression,3.0,Validation,1459.844019,1224.902636,10.413846
9,Fourier Regression,3.0,Test,2093.209849,1665.875976,11.212731


**EXPERIMENT 2: LSTM FOURIER ABLATION**

In [21]:

print("\n" + "=" * 65)
print("LSTM FOURIER ABLATION EXPERIMENTS")
print("=" * 65)

experiment_results = []

trained_models = {}

all_histories = {}

for K in K_VALUES:

    for seed in SEEDS:

        print(
            f"\nTraining LSTM | K={K} | Seed={seed}"
        )

        result = train_lstm(
            K=K,
            seed=seed,
            fit_end=train_end,
            validation_end=val_end,
            epochs=EPOCHS,
            verbose=False
        )

        model = result["model"]

        prep = result["prep"]

        history = result["history"]

        best_epoch = result["best_epoch"]

        _, val_predictions, val_target_idx = predict_model(
            model,
            result["features"],
            prep,
            train_end,
            val_end
        )

        val_actual = sales[
            val_target_idx
        ]

        metrics = calculate_metrics(
            val_actual,
            val_predictions
        )

        experiment_results.append({
            "Model": "LSTM + Fourier",
            "K": K,
            "Seed": seed,
            "Best Epoch": best_epoch,
            "Validation Loss": result["best_val_loss"],
            **metrics
        })

        trained_models[(K, seed)] = result

        all_histories[(K, seed)] = history

        print(
            f"Validation RMSE: {metrics['RMSE']:.2f} | "
            f"MAE: {metrics['MAE']:.2f} | "
            f"MAPE: {metrics['MAPE']:.2f}% | "
            f"Best Epoch: {best_epoch}"
        )


experiment_df = pd.DataFrame(
    experiment_results
)

print("\nIndividual experiment results:")

display(experiment_df)




LSTM FOURIER ABLATION EXPERIMENTS

Training LSTM | K=0 | Seed=42
Validation RMSE: 1669.98 | MAE: 1227.45 | MAPE: 11.00% | Best Epoch: 1

Training LSTM | K=0 | Seed=123
Validation RMSE: 1721.15 | MAE: 1275.78 | MAPE: 11.46% | Best Epoch: 1

Training LSTM | K=0 | Seed=2026
Validation RMSE: 1658.62 | MAE: 1217.40 | MAPE: 10.91% | Best Epoch: 1

Training LSTM | K=1 | Seed=42
Validation RMSE: 729.99 | MAE: 534.77 | MAPE: 4.47% | Best Epoch: 146

Training LSTM | K=1 | Seed=123
Validation RMSE: 737.60 | MAE: 558.67 | MAPE: 4.66% | Best Epoch: 148

Training LSTM | K=1 | Seed=2026
Validation RMSE: 709.56 | MAE: 523.56 | MAPE: 4.35% | Best Epoch: 146

Training LSTM | K=2 | Seed=42
Validation RMSE: 724.22 | MAE: 544.77 | MAPE: 4.57% | Best Epoch: 138

Training LSTM | K=2 | Seed=123
Validation RMSE: 671.14 | MAE: 506.71 | MAPE: 4.19% | Best Epoch: 147

Training LSTM | K=2 | Seed=2026
Validation RMSE: 692.37 | MAE: 516.86 | MAPE: 4.31% | Best Epoch: 146

Training LSTM | K=3 | Seed=42
Validation RM

,Model,K,Seed,Best Epoch,Validation Loss,RMSE,MAE,MAPE
0,LSTM + Fourier,0,42,1,1.311809,1669.976691,1227.451202,11.003803
1,LSTM + Fourier,0,123,1,1.385153,1721.150907,1275.776254,11.455713
2,LSTM + Fourier,0,2026,1,1.295698,1658.620942,1217.402563,10.908304
3,LSTM + Fourier,1,42,146,0.228011,729.985903,534.769705,4.465479
4,LSTM + Fourier,1,123,148,0.233344,737.601456,558.671665,4.661540
5,LSTM + Fourier,1,2026,146,0.215096,709.556783,523.555340,4.349048
6,LSTM + Fourier,2,42,138,0.227864,724.221773,544.772035,4.573058
7,LSTM + Fourier,2,123,147,0.192105,671.139160,506.707953,4.187111
8,LSTM + Fourier,2,2026,146,0.206726,692.365176,516.857627,4.309775
9,LSTM + Fourier,3,42,108,0.147676,592.950215,490.740265,3.958531


**ABLATION SUMMARY**

In [22]:

ablation_summary = (
    experiment_df
    .groupby("K")
    .agg({
        "RMSE": ["mean", "std"],
        "MAE": ["mean", "std"],
        "MAPE": ["mean", "std"],
        "Best Epoch": "mean"
    })
)

ablation_summary.columns = [
    "_".join(col)
    for col in ablation_summary.columns
]

ablation_summary = ablation_summary.reset_index()

ablation_summary = ablation_summary.sort_values(
    "RMSE_mean"
)

print("\nFourier ablation summary:")

display(ablation_summary)


Fourier ablation summary:


,K,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,Best Epoch_mean
5,5,516.729661,27.048998,429.469458,22.114390,3.523760,0.186253,98.000000
4,4,522.499988,20.197528,441.859892,11.872148,3.620820,0.117654,147.666667
3,3,586.786417,61.183891,478.037934,38.881120,3.918290,0.283621,125.000000
2,2,695.908703,26.718129,522.779205,19.710845,4.356648,0.197197,143.666667
1,1,725.714714,14.502007,538.998903,17.936099,4.492022,0.157928,146.666667
0,0,1683.249513,33.311034,1240.210006,31.208368,11.122607,0.292403,1.000000


**SELECT BEST FOURIER CONFIGURATION**

In [23]:

best_K = int(
    ablation_summary.iloc[0]["K"]
)

print(
    "\nSelected Fourier harmonics:",
    best_K
)

print(
    "Selection criterion: Lowest mean validation RMSE."
)


Selected Fourier harmonics: 5
Selection criterion: Lowest mean validation RMSE.


**ROLLING-ORIGIN VALIDATION**

In [24]:

rolling_results = []

if RUN_ROLLING_VALIDATION:

    print("\n" + "=" * 65)
    print("ROLLING-ORIGIN VALIDATION")
    print("=" * 65)

    origins = np.linspace(
        int(dev_end * 0.60),
        int(dev_end * 0.80),
        ROLLING_FOLDS
    ).astype(int)

    for fold, origin in enumerate(
        origins,
        start=1
    ):

        fold_end = min(
            origin + ROLLING_HORIZON,
            dev_end
        )

        if origin <= LOOKBACK + 10 or fold_end <= origin:

            continue

        print(
            f"\nFold {fold}: "
            f"Train < {dates[origin].date()} | "
            f"Validation through {dates[fold_end-1].date()}"
        )

        fold_result = train_lstm(
            K=best_K,
            seed=42 + fold,
            fit_end=origin,
            validation_end=fold_end,
            epochs=EPOCHS,
            verbose=False
        )

        _, predictions, fold_indices = predict_model(
            fold_result["model"],
            fold_result["features"],
            fold_result["prep"],
            origin,
            fold_end
        )

        metrics = calculate_metrics(
            sales[fold_indices],
            predictions
        )

        rolling_results.append({
            "Fold": fold,
            "K": best_K,
            "Train End": str(dates[origin-1].date()),
            "Validation Start": str(dates[origin].date()),
            "Validation End": str(dates[fold_end-1].date()),
            "Best Epoch": fold_result["best_epoch"],
            **metrics
        })

        print(
            f"Fold RMSE: {metrics['RMSE']:.2f} | "
            f"MAE: {metrics['MAE']:.2f}"
        )

rolling_df = pd.DataFrame(
    rolling_results
)

if len(rolling_df):

    print("\nRolling validation results:")

    display(rolling_df)

    print(
        "Mean rolling RMSE:",
        round(rolling_df["RMSE"].mean(), 2)
    )


ROLLING-ORIGIN VALIDATION

Fold 1: Train < 2008-11-01 | Validation through 2009-10-01
Fold RMSE: 676.03 | MAE: 590.19

Fold 2: Train < 2011-09-01 | Validation through 2012-08-01
Fold RMSE: 328.54 | MAE: 288.16

Fold 3: Train < 2014-07-01 | Validation through 2015-06-01
Fold RMSE: 607.35 | MAE: 511.99

Rolling validation results:


,Fold,K,Train End,Validation Start,Validation End,Best Epoch,RMSE,MAE,MAPE
0,1,5,2008-10-01,2008-11-01,2009-10-01,28,676.026450,590.186282,6.554123
1,2,5,2011-08-01,2011-09-01,2012-08-01,89,328.539157,288.162863,2.872220
2,3,5,2014-06-01,2014-07-01,2015-06-01,67,607.350848,511.992287,4.706116


Mean rolling RMSE: 537.31


**FINAL MODEL: REFIT ON TRAIN + VALIDATION**

In [25]:

print("\n" + "=" * 65)
print("FINAL MODEL TRAINING")
print("=" * 65)

selected_runs = [
    trained_models[(best_K, seed)]
    for seed in SEEDS
]

best_epochs = [
    r["best_epoch"]
    for r in selected_runs
    if r["best_epoch"] > 0
]

final_epochs = max(
    1,
    int(np.median(best_epochs))
)

print("Selected K:", best_K)

print("Final training epochs:", final_epochs)

print("Training on train + validation data.")

final_result = train_lstm(
    K=best_K,
    seed=42,
    fit_end=dev_end,
    validation_end=dev_end,
    fixed_epochs=final_epochs,
    verbose=True
)

final_model = final_result["model"]

final_prep = final_result["prep"]

final_features = final_result["features"]




FINAL MODEL TRAINING
Selected K: 5
Final training epochs: 99
Training on train + validation data.


**FINAL TEST EVALUATION**

In [26]:
print("\n" + "=" * 65)
print("FINAL UNTOUCHED TEST EVALUATION")
print("=" * 65)

_, final_predictions, final_target_indices = predict_model(
    final_model,
    final_features,
    final_prep,
    test_start,
    n
)

y_test_actual = sales[
    final_target_indices
]

test_dates = dates[
    final_target_indices
]

final_metrics = calculate_metrics(
    y_test_actual,
    final_predictions
)

print("\nFINAL LSTM + FOURIER RESULTS")

for metric, value in final_metrics.items():

    print(
        f"{metric}: {value:,.4f}"
    )




FINAL UNTOUCHED TEST EVALUATION

FINAL LSTM + FOURIER RESULTS
RMSE: 824.8303
MAE: 677.7435
MAPE: 4.3755


**FINAL TEST BASELINE COMPARISON**

In [27]:

test_comparison = []


naive_predictions = seasonal_naive(
    sales,
    final_target_indices
)

test_comparison.append({
    "Model": "Seasonal Naive",
    **calculate_metrics(
        y_test_actual,
        naive_predictions
    )
})




for K in K_VALUES:

    pred, _ = fourier_regression(
        K,
        dev_end,
        final_target_indices
    )

    test_comparison.append({
        "Model": f"Fourier Regression K={K}",
        **calculate_metrics(
            y_test_actual,
            pred
        )
    })




vanilla_result = train_lstm(
    K=0,
    seed=42,
    fit_end=dev_end,
    validation_end=dev_end,
    fixed_epochs=final_epochs
)

_, vanilla_predictions, vanilla_indices = predict_model(
    vanilla_result["model"],
    vanilla_result["features"],
    vanilla_result["prep"],
    test_start,
    n
)

test_comparison.append({
    "Model": "Vanilla LSTM",
    **calculate_metrics(
        sales[vanilla_indices],
        vanilla_predictions
    )
})




test_comparison.append({
    "Model": f"Hybrid LSTM + Fourier K={best_K}",
    **final_metrics
})


test_comparison_df = pd.DataFrame(
    test_comparison
).sort_values("RMSE")

print("\nFINAL TEST MODEL COMPARISON")

display(test_comparison_df)




FINAL TEST MODEL COMPARISON


,Model,RMSE,MAE,MAPE
8,Hybrid LSTM + Fourier K=5,824.830266,677.743471,4.375510
0,Seasonal Naive,1120.557406,861.033333,5.613525
6,Fourier Regression K=5,1258.870089,1019.133383,6.700355
5,Fourier Regression K=4,1283.571170,998.191556,6.628775
7,Vanilla LSTM,1507.200164,1188.981799,7.965489
4,Fourier Regression K=3,1513.161423,1214.794017,8.115455
3,Fourier Regression K=2,1586.667881,1301.016753,8.719930
2,Fourier Regression K=1,1752.830957,1394.414027,9.440095
1,Fourier Regression K=0,1938.307729,1516.146152,10.564408


**ACTUAL VS PREDICTED**

In [28]:

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=test_dates,
        y=y_test_actual,
        mode="lines+markers",
        name="Actual Sales",
        line=dict(width=2.5)
    )
)

fig.add_trace(
    go.Scatter(
        x=test_dates,
        y=final_predictions,
        mode="lines+markers",
        name="Hybrid LSTM Prediction",
        line=dict(
            width=2.5,
            dash="dash"
        )
    )
)

fig.update_layout(
    title=(
        f"Actual vs Predicted Sales "
        f"| K={best_K} "
        f"| RMSE={final_metrics['RMSE']:.2f}"
    ),
    xaxis_title="Date",
    yaxis_title="Alcohol Sales",
    template="plotly_white",
    hovermode="x unified",
    height=500
)

fig.show()



**TRAINING HISTORY**

In [29]:

history = final_result["history"]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=history["train"],
        name="Training Loss"
    )
)

fig.update_layout(
    title="Final Model Training Loss",
    xaxis_title="Epoch",
    yaxis_title="Residual MSE",
    template="plotly_white",
    height=400
)

fig.show()


**VALIDATION ABLATION VISUALIZATION**

In [30]:

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ablation_summary["K"],
        y=ablation_summary["RMSE_mean"],
        mode="lines+markers",
        name="Mean Validation RMSE"
    )
)

fig.update_layout(
    title="Effect of Fourier Harmonics on LSTM Performance",
    xaxis_title="Number of Fourier Harmonics (K)",
    yaxis_title="Mean Validation RMSE",
    template="plotly_white",
    height=450
)

fig.show()

**RESIDUAL ANALYSIS**

In [31]:

prediction_errors = (
    y_test_actual - final_predictions
)

fig = sp.make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        "Prediction Error Over Time",
        "Prediction Error Distribution"
    )
)

fig.add_trace(
    go.Scatter(
        x=test_dates,
        y=prediction_errors,
        mode="lines+markers",
        name="Error"
    ),
    row=1,
    col=1
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="red",
    row=1,
    col=1
)

fig.add_trace(
    go.Histogram(
        x=prediction_errors,
        name="Error Distribution"
    ),
    row=2,
    col=1
)

fig.update_layout(
    title="Final Test Residual Analysis",
    template="plotly_white",
    height=650
)

fig.show()



**RESIDUAL AUTOCORRELATION**

In [32]:
def calculate_acf(values, max_lag=24):

    values = np.asarray(values)

    values = values - values.mean()

    denominator = np.dot(
        values,
        values
    )

    if denominator == 0:
        return np.zeros(max_lag + 1)

    return np.array([
        np.dot(
            values[:-lag] if lag else values,
            values[lag:] if lag else values
        ) / denominator
        for lag in range(max_lag + 1)
    ])


max_lag = min(
    24,
    len(prediction_errors) - 1
)

acf_values = calculate_acf(
    prediction_errors,
    max_lag
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=np.arange(max_lag + 1),
        y=acf_values,
        name="Error ACF"
    )
)

fig.update_layout(
    title="Autocorrelation of Test Prediction Errors",
    xaxis_title="Lag (months)",
    yaxis_title="Autocorrelation",
    template="plotly_white",
    height=400
)

fig.show()



**EXPORT EXPERIMENT RESULTS**

In [33]:

experiment_df.to_csv(
    "lstm_fourier_experiments.csv",
    index=False
)

ablation_summary.to_csv(
    "fourier_ablation_summary.csv",
    index=False
)

baseline_df.to_csv(
    "baseline_validation_test.csv",
    index=False
)

test_comparison_df.to_csv(
    "final_model_comparison.csv",
    index=False
)

if len(rolling_df):

    rolling_df.to_csv(
        "rolling_validation_results.csv",
        index=False
    )



**SAVE FINAL MODEL CHECKPOINT**

In [34]:

checkpoint = {

    "model_state_dict": final_model.state_dict(),

    "model_config": {

        "input_size": final_features.shape[1],

        "hidden_size": HIDDEN_SIZE,

        "num_layers": NUM_LAYERS,

        "dropout": DROPOUT
    },

    "lookback": LOOKBACK,

    "fourier_K": best_K,

    "fourier_period": 12,

    "trend_coef": final_prep["coef"],

    "trend_min": final_prep["trend_min"],

    "trend_max": final_prep["trend_max"],

    "res_scaler_mean": final_prep["scaler"].mean_,

    "res_scaler_scale": final_prep["scaler"].scale_,

    "final_epochs": final_epochs,

    "seed": 42,

    "feature_order": [
        "residual_scaled"
    ] + [
        name
        for k in range(1, best_K + 1)
        for name in (
            f"sin_{k}",
            f"cos_{k}"
        )
    ] + [
        "trend_normalized"
    ],

    "test_metrics": final_metrics
}

torch.save(
    checkpoint,
    "best_lstm_fourier_v2.pth"
)



**FINAL SUMMARY**

In [35]:
print("\n" + "=" * 65)
print("EXPERIMENT COMPLETED")
print("=" * 65)

print("Selected Fourier K:", best_K)

print("Lookback:", LOOKBACK)

print("Hidden Size:", HIDDEN_SIZE)

print("LSTM Layers:", NUM_LAYERS)

print("Final Epochs:", final_epochs)

print("\nFinal Test Metrics:")

for key, value in final_metrics.items():

    print(f"{key}: {value:.4f}")

print("\nSaved files:")

print("1. best_lstm_fourier_v2.pth")

print("2. lstm_fourier_experiments.csv")

print("3. fourier_ablation_summary.csv")

print("4. baseline_validation_test.csv")

print("5. final_model_comparison.csv")

if len(rolling_df):

    print("6. rolling_validation_results.csv")

print("\nAll experiments completed successfully.")


EXPERIMENT COMPLETED
Selected Fourier K: 5
Lookback: 36
Hidden Size: 64
LSTM Layers: 2
Final Epochs: 99

Final Test Metrics:
RMSE: 824.8303
MAE: 677.7435
MAPE: 4.3755

Saved files:
1. best_lstm_fourier_v2.pth
2. lstm_fourier_experiments.csv
3. fourier_ablation_summary.csv
4. baseline_validation_test.csv
5. final_model_comparison.csv
6. rolling_validation_results.csv

All experiments completed successfully.
